In [ ]:
# !pip install wikipedia

## Wikipedia Retriever

In [5]:
from langchain_community.retrievers import WikipediaRetriever

In [11]:
query = 'Langchain'

retriever = WikipediaRetriever(top_k_results=3, lang='en')
answer = retriever.invoke(query)

In [12]:
answer[0].page_content

'LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain\'s use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.\n\n\n== History ==\nLangChain was launched in October 2022 as an open source project by Harrison Chase, while working at machine learning startup Robust Intelligence. In April 2023, LangChain had incorporated and the new startup raised over $20 million in funding at a valuation of at least $200 million from venture firm Sequoia Capital, a week after announcing a $10 million seed investment from Benchmark.\nIn the third quarter of 2023, the LangChain Expression Language (LCEL) was introduced, which provides a declarative way to define chains of actions.\nIn October 2023 LangChain introduced LangServe, a deployment tool to host LCEL code as a production-ready API.\nIn F

In [ ]:
for i, doc in enumerate(answer):
    print(f'\n Result {i+1}')
    print(doc.page_content)

## Vector Store Retriever

In [14]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [16]:
embeddings = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')

c:\Users\ManishAgrahari\Desktop\my_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5340.75it/s]


In [15]:
documents = [
    Document(
        page_content="LangChain is a framework for building applications powered by language models.",
        metadata={"source": "LangChain overview", "topic": "LangChain"},
    ),
    Document(
        page_content="Retrieval augmented generation (RAG) retrieves relevant documents and uses them as context for a language model.",
        metadata={"source": "RAG overview", "topic": "RAG"},
    ),
    Document(
        page_content="A retriever searches a collection of documents and returns the ones most relevant to a query.",
        metadata={"source": "Retriever overview", "topic": "Retrievers"},
    ),
]

In [24]:
vector_store = Chroma.from_documents(documents=documents, embedding=embeddings, collection_name='new_collection')

vec_retriever = vector_store.as_retriever(search_kwargs={'k': 1})

In [25]:
query = 'what retriever do?'

answer = vec_retriever.invoke(query)

for i, doc in enumerate(answer):
    print(f'\nResult {i+1}')
    print(doc.page_content)


Result 1
A retriever searches a collection of documents and returns the ones most relevant to a query.


## Search Staryegy Retriever
1. Maximum marginal relevance MMR
2. Multi Query 
3. Contexual Compresor

In [29]:
from langchain_community.vectorstores import FAISS

query = "What are the Renewable Energy Sources?"

docs = [
    "Solar power converts sunlight into electricity.",
    "Solar panels are widely used in residential homes.",
    "Wind turbines harness wind energy for electricity.",
    "Hydropower generates energy from flowing water.",
    "Geothermal energy comes from heat inside the Earth."
]

In [34]:
faiss_vector_store = FAISS.from_texts(docs, embeddings)

fasis_ret = faiss_vector_store.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.5})

In [35]:
results = fasis_ret.invoke(query)

for i, doc in enumerate(results):
    print(f'\nMMR Result {i+1}')
    print(doc.page_content)


MMR Result 1
Wind turbines harness wind energy for electricity.

MMR Result 2
Geothermal energy comes from heat inside the Earth.

MMR Result 3
Solar panels are widely used in residential homes.


## Multi Query

In [37]:
!pip install langchain_groq

  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [39]:
texts = [
    "RAG retrieves relevant documents and uses them to help a language model answer questions.",
    "A retriever searches a collection and returns documents related to a query.",
    "Embeddings represent text as vectors so similar meanings can be compared.",
    "FAISS searches vectors to find similar documents.",
    "Multi-query retrieval rewrites a question in several ways to find more relevant documents.",
    "The water cycle includes evaporation, condensation, and precipitation.",
]

In [40]:
vector_store = FAISS.from_texts(texts, embeddings)

In [51]:
load_dotenv()

llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0.7)

In [53]:
multi_retriever = MultiQueryRetriever.from_llm(retriever=vector_store.as_retriever(), llm=llm)

In [54]:
query = "How does a retriever find relevant information for a RAG system?"

mul_result = multi_retriever.invoke(query)

for i, doc in enumerate(mul_result):
    print(f'\nResult {i+1}')
    print(doc.page_content)


Result 1
A retriever searches a collection and returns documents related to a query.

Result 2
RAG retrieves relevant documents and uses them to help a language model answer questions.

Result 3
Multi-query retrieval rewrites a question in several ways to find more relevant documents.

Result 4
FAISS searches vectors to find similar documents.

Result 5
Embeddings represent text as vectors so similar meanings can be compared.


## Contexual Compresor

In [55]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [56]:
query = 'How do I reset my router password'

docs = [
    "To reset your router password, connect to the router and open its admin page in a browser. Sign in, find the Wi-Fi or wireless settings, change the password, and save the changes.",
    "If you cannot sign in to the router admin page, check the label or manual for the default address and login details. If those were changed, you may need to factory-reset the router and set it up again.",
    "Use a strong Wi-Fi password with a mix of letters, numbers, and symbols. After changing it, reconnect your devices using the new password.",
    "A router sends data between your home network and the internet. Restarting it can fix some connection problems, but it does not usually change the Wi-Fi password.",
    "The water cycle includes evaporation, condensation, and precipitation. Sunlight warms water on Earth’s surface and causes evaporation.",
]

In [57]:
vector_store = FAISS.from_texts(docs, embeddings)

llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0.7)

In [58]:
base_retriever = vector_store.as_retriever()

In [59]:
compressor = LLMChainExtractor.from_llm(llm)

In [60]:
compressor_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=base_retriever, )

In [61]:
res = compressor_retriever.invoke(query)

In [62]:
for i, doc in enumerate(res):
    print(f'\nResult {i+1}')
    print(doc.page_content)


Result 1
To reset your router password, connect to the router and open its admin page in a browser. Sign in, find the Wi-Fi or wireless settings, change the password, and save the changes.

Result 2
If you cannot sign in to the router admin page, check the label or manual for the default address and login details. If those were changed, you may need to factory-reset the router and set it up again.

Result 3
Restarting it can fix some connection problems, but it does not usually change the Wi‑Fi password.

Result 4
Use a strong Wi-Fi password with a mix of letters, numbers, and symbols. After changing it, reconnect your devices using the new password.
